# The Orthogonalisation Trick: Manual DML Implementation

**Goal:** Implement DML from scratch and see how ML residualisation recovers causal effects with nonlinear confounding.

**Time:** 15 minutes

You will build the residual-on-residual regression step by step, compare it to OLS,
and apply it to a weather shock / natural gas basis scenario.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## Simulate: Weather Shocks on Natural Gas Basis

The DGP has nonlinear confounding -- sin, squared, and interaction terms.
OLS with linear controls will miss these patterns.

In [ ]:
n = 2000
p = 50
true_theta = 0.8

X = np.random.randn(n, p)  # Storage, pipeline, demand, HDD, etc.

# Treatment: weather shock (nonlinear function of seasonal factors)
D = np.sin(X[:, 0]) + 0.5 * X[:, 1] ** 2 + np.random.randn(n) * 0.3

# Outcome: basis spread (nonlinear confounding)
Y = (true_theta * D + np.exp(0.3 * X[:, 0])
     + 0.5 * np.abs(X[:, 2]) + 0.3 * X[:, 3] * X[:, 4]
     + np.random.randn(n) * 0.5)

print(f'n={n}, p={p}')
print(f'True causal effect: {true_theta}')
print(f'Nonlinear confounders: sin(X0), X1^2, exp(X0), |X2|, X3*X4')

## Step 1: OLS Baseline (Linear Controls)

Run standard OLS with all 50 controls. Since confounding is nonlinear, OLS will be biased.

In [ ]:
DX = np.column_stack([D, X])
ols_model = sm.OLS(Y, sm.add_constant(DX)).fit()

print(f'True effect: {true_theta:.2f}')
print(f'OLS estimate: {ols_model.params[1]:.2f}')
print(f'OLS SE: {ols_model.bse[1]:.3f}')
print(f'OLS bias: {ols_model.params[1] - true_theta:+.3f}')
print(f'\nOLS misses nonlinear confounding and is biased.')

## Step 2: Manual DML Implementation

Implement the residual-on-residual regression with cross-fitting.

In [ ]:
def manual_dml(Y, D, X, ml_model_y=None, ml_model_d=None, n_folds=5):
    """Manual DML with cross-fitting."""
    n = len(Y)
    resid_Y = np.zeros(n)
    resid_D = np.zeros(n)

    if ml_model_y is None:
        ml_model_y = RandomForestRegressor(n_estimators=100, random_state=42)
    if ml_model_d is None:
        ml_model_d = RandomForestRegressor(n_estimators=100, random_state=42)

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

    for train_idx, test_idx in kf.split(X):
        # Clone models for each fold
        from sklearn.base import clone
        rf_y = clone(ml_model_y)
        rf_d = clone(ml_model_d)

        rf_y.fit(X[train_idx], Y[train_idx])
        rf_d.fit(X[train_idx], D[train_idx])

        resid_Y[test_idx] = Y[test_idx] - rf_y.predict(X[test_idx])
        resid_D[test_idx] = D[test_idx] - rf_d.predict(X[test_idx])

    # Treatment effect
    theta = np.sum(resid_D * resid_Y) / np.sum(resid_D ** 2)

    # Robust standard error
    epsilon = resid_Y - theta * resid_D
    se = np.sqrt(np.mean(resid_D**2 * epsilon**2) /
                 (np.mean(resid_D**2)**2) / n)

    return theta, se, resid_Y, resid_D

# Run DML
theta_dml, se_dml, rY, rD = manual_dml(Y, D, X)

print(f'True effect:   {true_theta:.2f}')
print(f'DML estimate:  {theta_dml:.2f}')
print(f'DML SE:        {se_dml:.3f}')
print(f'95% CI:        [{theta_dml - 1.96*se_dml:.3f}, {theta_dml + 1.96*se_dml:.3f}]')
print(f'DML bias:      {theta_dml - true_theta:+.3f}')

## Visualise: Residual Scatter Plots

Compare OLS residuals (linear first stage) vs DML residuals (ML first stage).

In [ ]:
# OLS residuals for comparison
ols_rY = sm.OLS(Y, sm.add_constant(X)).fit().resid
ols_rD = sm.OLS(D, sm.add_constant(X)).fit().resid
ols_fwl = sm.OLS(ols_rY, sm.add_constant(ols_rD)).fit()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: OLS (FWL) residuals
axes[0].scatter(ols_rD, ols_rY, alpha=0.2, s=8, color='steelblue')
xr = np.linspace(ols_rD.min(), ols_rD.max(), 100)
axes[0].plot(xr, ols_fwl.params[0] + ols_fwl.params[1] * xr,
             color='red', linewidth=2, label=f'slope = {ols_fwl.params[1]:.2f}')
axes[0].set_title('FWL: Linear Residuals (biased)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('D residuals', fontsize=11)
axes[0].set_ylabel('Y residuals', fontsize=11)
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Right: DML residuals
axes[1].scatter(rD, rY, alpha=0.2, s=8, color='forestgreen')
xr_ml = np.linspace(rD.min(), rD.max(), 100)
axes[1].plot(xr_ml, theta_dml * xr_ml,
             color='red', linewidth=2, label=f'slope = {theta_dml:.2f}')
axes[1].set_title('DML: ML Residuals (unbiased)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('D residuals', fontsize=11)
axes[1].set_ylabel('Y residuals', fontsize=11)
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.suptitle(f'True effect = {true_theta}', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Compare ML Models for First Stage

Try different ML models in the first stage: linear regression, random forest, and gradient boosting.

In [ ]:
models = {
    'Linear (FWL)': (LinearRegression(), LinearRegression()),
    'Random Forest': (RandomForestRegressor(100, random_state=42),
                      RandomForestRegressor(100, random_state=42)),
    'Gradient Boosting': (GradientBoostingRegressor(100, random_state=42),
                          GradientBoostingRegressor(100, random_state=42)),
}

print(f'True effect: {true_theta:.2f}')
print(f'{"Model":<25} {"Estimate":>8} {"SE":>8} {"Bias":>8}')
print('=' * 55)

for name, (m_y, m_d) in models.items():
    t, s, _, _ = manual_dml(Y, D, X, ml_model_y=m_y, ml_model_d=m_d)
    print(f'{name:<25} {t:>8.3f} {s:>8.3f} {t - true_theta:>+8.3f}')

print()
print('Linear first stage is biased (misses nonlinear confounding).')
print('RF and GBM recover the true effect.')

## Summary

**What you learned:**
1. Robinson's partially linear model decomposes causal estimation into residualisation + regression
2. DML replaces OLS first stages with ML to handle nonlinear confounding
3. The manual DML implementation is about 30 lines of Python
4. ML first stages (RF, GBM) outperform linear first stages when confounding is nonlinear

**What is next:**
- **Module 03:** Neyman orthogonal scores -- why DML is robust to first-stage ML errors
- **Module 04:** Cross-fitting -- why out-of-sample predictions are essential

**Key takeaway:** The treatment effect is the slope of the residual-on-residual regression.
Better ML in the first stage means cleaner residuals and a more accurate treatment effect.

In [ ]:
learning_objectives(["Robinson's partially linear model decomposes causal estimation into residualisation + regression", "DML replaces OLS first stages with ML to handle nonlinear confounding", "The manual DML implementation is about 30 lines of Python", "ML first stages (RF, GBM) outperform linear first stages when confounding is nonlinear"])

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])